In [35]:
import pandas as pd
import numpy as np
import argparse
import pyranges as pr
from gtfparse import read_gtf #initially tested with version 1.3.0)
from tqdm import tqdm
import time
import warnings
import glob 
import time
import gzip
from pathlib import Path
import concurrent.futures
import concurrent.futures
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore", category=FutureWarning, module="pyranges")

parser = argparse.ArgumentParser(description='Read in file that lists junctions for all samples, \
                                 one file per line and no header')

parser.add_argument('--junc_files', dest='junc_files',
                    help='path that has all junction files along with counts in single cells or bulk samples, \
                    make sure path ends in "/" Can also be a comma seperated list of paths. If you have a complex folder structure, \
                        provide the most root folder that contains all the junction files. The script will recursively search for junction files with the suffix provided in the next argument.')

parser.add_argument('--sequencing_type', dest='sequencing_type',
                    default='single_cell',
                    help='were the junction obtained using data from single cell or bulk sequencing? \
                        options are "single_cell" or "bulk". Default is "single_cell"')

parser.add_argument('--gtf_file', dest='gtf_file', 
                    default = None,
                    help='a path to a gtf file to annotate high confidence junctions, \
                    ideally from long read sequencing, if not provided, then the script will not \
                        annotate junctions based on gtf file')

parser.add_argument('--output_file', dest='output_file', 
                    default='intron_clusters.txt',
                    help='name of the output file to save intron cluster file to')

parser.add_argument('--junc_bed_file', dest='junc_bed_file', 
                    default='juncs.bed',
                    help='name of the output bed file to save final list of junction coordinates to')

parser.add_argument('--threshold_inc', dest='threshold_inc',
                    default=0.005,
                    help='threshold to use for removing clusters that have junctions with low read counts \
                        (proportion of reads relative to intron cluster) at either end, default is 0.01')

parser.add_argument('--min_intron_length', dest='min_intron_length',
                    default=50,
                    help='minimum intron length to consider, default is 50')

parser.add_argument('--max_intron_length', dest='max_intron_length',
                    default=500000,
                    help='maximum intron length to consider, default is 500000')

parser.add_argument('--min_junc_reads', dest='min_junc_reads',
                    default=1,
                    help='minimum number of reads to consider a junction, default is 1')

parser.add_argument('--keep_singletons', dest='keep_singletons', 
                    default=False,
                    help='Indicate whether you would like to keep "clusters" composed of just one junction.\
                          Default is False which means do not keep singletons')

parser.add_argument('--junc_suffix', dest='junc_suffix', #set default param to *.junc, 
                    default='*.juncs', 
                    help='suffix of junction files')

parser.add_argument('--min_num_cells_wjunc', dest='min_num_cells_wjunc',
                    default=1,
                    help='minimum number of cells that have a junction to consider it, default is 1')

parser.add_argument('--run_notebook', dest='run_notebook',
                    default=False,
                    help='Indicate whether you would like to run the script in a notebook and return the table in session.\
                          Default is False')

parser.add_argument('--filter_shared_ss', dest='filter_shared_ss',
                    default=True,
                    help='Indicate whether you would like to filter clusters by junctions with shared splice sites.\
                          Default is True')

args = parser.parse_args(args=[])

#+++++++++++++++++++++++++++++++++++++++++++++++++++++++
#                      Utilities
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++

def process_gtf(gtf_file): #make this into a seperate script that processes the gtf file into gr object that can be used in the main scriptas input 
    """
    Process the GTF file into a pyranges object.

    Parameters:
    - gtf_file (str): Path to the GTF file.

    Returns:
    - gtf_exons_gr (pyranges.GenomicRanges): Processed pyranges object.
    """

    print("The gtf file you provided is " + gtf_file)
    print("Reading the gtf may take a minute...")

    # calculate how long it takes to read gtf_file and report it 
    start_time = time.time()
    #[1] extract all exons from gtf file provided 
    gtf = read_gtf(gtf_file, result_type="pandas") #to reduce the speed of this, can just get rows with exon in the feature column (preprocess this before running package)? check if really necessary
    end_time = time.time()

    print("Reading gtf file took " + str(round((end_time-start_time), 2)) + " seconds")
    # assert that gtf is a non empty dataframe otherwise return an error
    if gtf.empty or type(gtf) != pd.DataFrame:
        raise ValueError("The gtf file provided is empty or not a pandas DataFrame. Please provide a valid gtf file and ensure you have the \
                         latest version of gtfparse installed by running 'pip install gtfparse --upgrade'")
    
    # Convert the seqname column to a string in gtf 
    gtf["seqname"] = gtf["seqname"].astype(str)

    # Make a copy of the DataFrame
    gtf_exons = gtf[(gtf["feature"] == "exon")].copy()

    if gtf_exons['seqname'].str.contains('chr').any():
        gtf_exons.loc[gtf_exons['seqname'].str.contains('chr'), 'seqname'] = gtf_exons['seqname'].map(lambda x: x.lstrip('chr').rstrip('chr'))

    if not set(['seqname', 'start', 'end', 'score', 'strand', 'gene_id', 'gene_name', 'transcript_id', 'exon_id']).issubset(gtf_exons.columns):
        # print the columns that the file is missing
        missing_cols = set(['seqname', 'start', 'end', 'score', 'strand', 'gene_id', 'gene_name', 'transcript_id', 'exon_id']).difference(gtf_exons.columns)
        print("Your gtf file is missing the following columns: " + str(missing_cols))

        # if the missing column is just exon_id, we can generate it
        if "exon_id" in missing_cols:
            # add exon_id to gtf_exons
            print("Adding exon_id column to gtf file")
            gtf_exons.loc[:, "exon_id"] = gtf_exons["transcript_id"] + "_" + gtf_exons["start"].astype(str) + "_" + gtf_exons["end"].astype(str)
        else:
            pass

    # Convert the DataFrame to a PyRanges object
    gtf_exons_gr = pr.from_dict({"Chromosome": gtf_exons["seqname"], "Start": gtf_exons["start"], "End": gtf_exons["end"], "Strand": gtf_exons["strand"], "gene_id": gtf_exons["gene_id"], "gene_name": gtf_exons["gene_name"], "transcript_id": gtf_exons["transcript_id"], "exon_id": gtf_exons["exon_id"]})

    # Remove rows where exon start and end are the same or when gene_name is empty
    gtf_exons_gr = gtf_exons_gr[ ~ (gtf_exons_gr.Start == gtf_exons_gr.End)]
    gtf_exons_gr = gtf_exons_gr[ ~ (gtf_exons_gr.gene_name == "")]

    # When do I need to do this? depends on gtf file used? base 0 or 1? probably need this to be a parameter 
    gtf_exons_gr.Start = gtf_exons_gr.Start-1

    # Drop duplicated positions on same strand 
    gtf_exons_gr = gtf_exons_gr.drop_duplicate_positions(strand=True) # Why are so many gone after this? 

    # Print the number of unique exons, transcript ids, and gene ids
    print("The number of unique exons is " + str(len(gtf_exons_gr.exon_id.unique())))
    print("The number of unique transcript ids is " + str(len(gtf_exons_gr.transcript_id.unique())))
    print("The number of unique gene ids is " + str(len(gtf_exons_gr.gene_id.unique())))
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++")
    return(gtf_exons_gr)

def filter_junctions_by_shared_splice_sites(df):
    """
    Filter junctions by shared splice sites.

    Parameters:
    - df (pd.DataFrame): Input DataFrame.

    Returns:
    - pd.DataFrame: Filtered DataFrame.
    """
    # Function to apply to each group (cluster)
    def filter_group(group):
        # Find duplicated start and end positions within the group
        duplicated_starts = group['Start'].duplicated(keep=False)
        duplicated_ends = group['End'].duplicated(keep=False)
        
        # Keep rows where either start or end position is duplicated (this results in at least two junctions in every cluster)
        return group[duplicated_starts | duplicated_ends]
    
    # Group by 'Cluster' and apply the filtering function
    filtered_df = df.groupby('Cluster').apply(filter_group).reset_index(drop=True)
    return filtered_df.Cluster.unique()

def read_junction_files(junc_files, junc_suffix):
    """
    Read junction files.

    Parameters:
    - junc_files (list): List of paths to junction files.
    - junc_suffix (str): Suffix of junction files.

    Returns:
    - pd.DataFrame: Concatenated DataFrame of junction files.
    """
    all_juncs_list = []

    for junc_path in tqdm(junc_files):
        junc_path = Path(junc_path)
        #print(f"Reading in junction files from {junc_path}")

        if junc_path.is_dir():
            junc_files_in_path = list(junc_path.rglob(junc_suffix))
            if not junc_files_in_path:
                print(f"No junction files found in {junc_path} with suffix {junc_suffix}")
                continue
        else:
            junc_files_in_path = [junc_path]

        #print(f"The number of junction files to be processed is {len(junc_files_in_path)}")

        files_not_read = []

        for junc_file in junc_files_in_path:
            try:
                juncs = pd.read_csv(junc_file, sep="\t", header=None)
                juncs['file_name'] = junc_file  # Add the file name as a new column
                juncs['cell_type'] = junc_file
                all_juncs_list.append(juncs)  # Append the DataFrame to the list
            except Exception as e:
                #print(f"Could not read in {junc_file}: {e}")
                files_not_read.append(junc_file)

    if len(files_not_read) > 0:
        print("The total number of files that could not be read is " + str(len(files_not_read)) + " as these had no junctions")

    # Concatenate all DataFrames into a single DataFrame
    all_juncs = pd.concat(all_juncs_list, ignore_index=True) if all_juncs_list else pd.DataFrame()

    return all_juncs

def clean_up_juncs(all_juncs, col_names, min_intron, max_intron):
    
    # Apply column names to the DataFrame
    all_juncs.columns = col_names
    
    # Split 'blockSizes' into two new columns and convert them to integers (this step takes a while)
    all_juncs[['block_add_start', 'block_subtract_end']] = all_juncs["blockSizes"].str.split(',', expand=True).astype(int)

    # Adjust 'chromStart' and 'chromEnd' based on 'block_add_start' and 'block_subtract_end'
    all_juncs["chromStart"] += all_juncs['block_add_start']
    all_juncs["chromEnd"] -= all_juncs['block_subtract_end']

    # Calculate 'intron_length' and filter based on 'min_intron' and 'max_intron'
    all_juncs["intron_length"] = all_juncs["chromEnd"] - all_juncs["chromStart"]
    mask = (all_juncs["intron_length"] >= min_intron) & (all_juncs["intron_length"] <= max_intron)
    all_juncs = all_juncs[mask]

    # Filter for 'chrom' column to handle "chr" prefix
    all_juncs = all_juncs.copy()

    # New filter for 'chrom' column to handle "chr" prefix, using .loc for safe in-place modification
    standard_chromosomes_pattern = r'^(?:chr)?(?:[1-9]|1[0-9]|2[0-2]|X|Y|MT)$'
    all_juncs = all_juncs[all_juncs['chrom'].str.match(standard_chromosomes_pattern)]

    print("Cleaning up 'chrom' column")
   
    # Remove "chr" prefix from 'chrom' column
    all_juncs['chrom'] = all_juncs['chrom'].str.replace(r'^chr', '', regex=True)
    
    # Add 'junction_id' column
    all_juncs['junction_id'] = all_juncs['chrom'] + '_' + all_juncs['chromStart'].astype(str) + '_' + all_juncs['chromEnd'].astype(str)
    
    # Get total score for each junction and merge with all_juncs with new column "total_counts"
    all_juncs = all_juncs.groupby('junction_id').agg({'score': 'sum'}).reset_index().merge(all_juncs, on='junction_id', how='left')

    # rename score_x and score_y to total_junc_counts and score 
    all_juncs.rename(columns={'score_x': 'counts_total', 'score_y': 'score'}, inplace=True)

    return(all_juncs)

def mapping_juncs_exons(juncs_gr, gtf_exons_gr, singletons):
    print("Annotating junctions with known exons based on input gtf file")
    
    # for each junction, the start of the junction should equal end of exons and end of junction should equal start of exon 
    juncs_gr = juncs_gr.k_nearest(gtf_exons_gr, strandedness = "same", ties="different", k=2, overlap=False)
    # ensure distance parameter is still 1 
    juncs_gr = juncs_gr[abs(juncs_gr.Distance) == 1]

    # group juncs_gr by gene_id and ensure that each junction has Start and End aligning with at least one End_b and Start_b respectively
    grouped_gr = juncs_gr.df.groupby("gene_id")
    juncs_keep = []
    for name, group in grouped_gr:
        group = group[(group.Start.isin(group.End_b)) & (group.End.isin(group.Start_b))]
        # save junctions that are found here after filtering for matching start and end positions
        juncs_keep.append(group.junction_id.unique())

    # flatten the list of lists
    juncs_keep = [item for sublist in juncs_keep for item in sublist]
    juncs_gr = juncs_gr[juncs_gr.junction_id.isin(juncs_keep)]
    
    print("The number of junctions after assessing distance to exons is " + str(len(juncs_gr.junction_id.unique())))
    if len(juncs_gr.junction_id.unique()) < 5000:
        print("There are less than 5000 junctions after assessing distance to exons. Please check your gtf file and ensure that it is in the correct format (start and end positions are not off by 1).", flush=True)
    
    print("Clustering intron splicing events by gene_id")
    juncs_coords_unique = juncs_gr[['Chromosome', 'Start', 'End', 'Strand', 'junction_id', 'gene_id']].drop_duplicate_positions()
    clusters = juncs_coords_unique.cluster(by="gene_id", slack=-1, count=True)
    print("The number of clusters after clustering by gene_id is " + str(len(clusters.Cluster.unique()))) 

    if singletons == False:
        # remove singletons 
        clusters = clusters[clusters.Count > 1]
        # update juncs_gr to only include junctions that are part of clusters
        juncs_gr = juncs_gr[juncs_gr.junction_id.isin(clusters.junction_id)]
        # update juncs_coords_unique to only include junctions that are part of clusters
        juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(clusters.junction_id)]
        print("The number of junctions after removing singletons is " + str(len(juncs_coords_unique.junction_id.unique())))
        print("The number of clusters after removing singletons is " + str(len(clusters.Cluster.unique()))) 
        return juncs_gr, juncs_coords_unique, clusters
    else:
        return juncs_gr, juncs_coords_unique, clusters

def basepair_to_kilobase(bp):
    return bp / 1000  # Convert base pairs to kilobases

def visualize_local_events(dat, junc_id=None, cluster_id=None, p_usage_ratio=True):

    # Filter dat based on junc_id, or cluster_id 
    if junc_id is not None:
        dat = dat[dat.Cluster == dat[dat.junction_id == junc_id].Cluster.values[0]]
    elif cluster_id is not None:
        dat = dat[dat.Cluster == cluster_id]

    # Get junctions
    juncs = dat[["chrom", "chromStart", "chromEnd", "strand", "intron_length", "counts_total", "Start_b", "End_b", "exon_id"]]
    juncs = juncs.drop_duplicates()
    juncs["junc_usage_ratio"] = juncs["counts_total"] / juncs["counts_total"].sum()

    # Sort junctions based on strand
    if juncs.strand.values[0] == "+":
        juncs = juncs.sort_values("chromStart")
    else:
        juncs = juncs.sort_values("chromEnd", ascending=False)

    # Convert genomic coordinates to kilobases
    juncs["chromStart_kb"] = basepair_to_kilobase(juncs["chromStart"])
    juncs["chromEnd_kb"] = basepair_to_kilobase(juncs["chromEnd"])
    # convert exon coordinates to kilobases
    juncs["Start_b"] = basepair_to_kilobase(juncs["Start_b"])
    juncs["End_b"] = basepair_to_kilobase(juncs["End_b"])

    print(juncs[["chrom", "chromStart_kb", "chromEnd_kb", "strand", "intron_length", "counts_total", "Start_b", "End_b", "exon_id"]])

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, len(juncs) * 0.5))

    # unique exons
    exon_ids = juncs.exon_id.unique()
    colors = plt.cm.tab20.colors
    color_dict = {exon_id: colors[i] for i, exon_id in enumerate(exon_ids)}
    cmap = plt.get_cmap('plasma')

    # Plot junctions as lines with varying colors based on usage ratio
    for i, (_, junc) in enumerate(juncs.iterrows()):
        color = cmap(junc["junc_usage_ratio"])
        ax.plot([junc["chromStart_kb"], junc["chromEnd_kb"]], [i, i], color=color)


    # Add vertical lines at unique start_b and end_b positions for each exon with dashed line
    for exon_id, group in juncs.groupby("exon_id"):
        for _, exon in group.iterrows():
            ax.axvline(x=exon["Start_b"], color = color_dict[exon_id], linestyle="--")
            ax.axvline(x=exon["End_b"], color = color_dict[exon_id], linestyle="--")

    # Set labels and title 
    ax.set_xlabel(f"Genomic Position on chr{juncs.chrom.values[0]} ({juncs.strand.values[0]}) [Kilobases]")
    ax.set_yticks([])  # Remove y-axis ticks
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0f}'.format(x)))  # Disable scientific notation

    # Plot junction usage ratios if requested
    if p_usage_ratio:
        for i, (_, junc) in enumerate(juncs.iterrows()):
            ax.text(junc["chromEnd_kb"], i, f'{junc["junc_usage_ratio"]:.3f}', verticalalignment='center', fontsize=8)

    # check if gene_name column is in dat and use it as title if it is otherwise use gene_id
    if "gene_name" in dat.columns:
        ax.set_title(f"Visualization of Junctions in Cluster {dat.Cluster.values[0]} in the Gene {dat.gene_name.values[0]}")
    else:
        ax.set_title(f"Visualization of Junctions in Cluster {dat.Cluster.values[0]} in the Gene {dat.gene_id.values[0]}")

    print("The junction of interest is " + junc_id)
    plt.show()

def refine_clusters(clust_info):
    # for all start positions that are same for each cluster get the sum counts_total
    clust_info_5ss = clust_info.groupby(['Cluster', 'Start']).agg({'counts_total': 'sum'}).reset_index()
    clust_info_3_ss = clust_info.groupby(['Cluster', 'End']).agg({'counts_total': 'sum'}).reset_index()
    # rename columns in 5ss to be total 5ss counts
    clust_info_5ss.rename(columns={'counts_total': 'total_5ss_counts'}, inplace=True)
    clust_info_3_ss.rename(columns={'counts_total': 'total_3ss_counts'}, inplace=True)
    # remove Start and End column from each
    clust_info = clust_info.merge(clust_info_5ss, on=['Cluster', 'Start'])
    clust_info = clust_info.merge(clust_info_3_ss, on=['Cluster', 'End'])

    # give each junction a 5ss fraction and 3ss fraction and then add column counts_total 
    clust_info['5SS_usage'] = clust_info['counts_total'] / clust_info['total_5ss_counts']
    clust_info['3SS_usage'] = clust_info['counts_total'] / clust_info['total_3ss_counts']
    clust_info["min_usage"] = clust_info[["5SS_usage", "3SS_usage"]].min(axis=1)
    print("Done refining clusters!")
    return(clust_info)

#+++++++++++++++++++++++++++++++++++++++++++++++++++++++
#        Run analysis and obtain intron clusters
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++

def main(junc_files, gtf_file, output_file, sequencing_type, junc_bed_file, threshold_inc, min_intron, max_intron, min_junc_reads, singleton, junc_suffix, min_num_cells_wjunc, filter_shared_ss, run_notebook):
    
    #1. Check format of junc_files and convert to list if necessary
    # Can either be a list of folders with junction files or a single folder with junction files

    # first check if junc_files is a list already 
    if type(junc_files) == list:
        # do nothing 
        pass
    elif "," in junc_files:
        junc_files = junc_files.split(",")
    else:
        # if junc_files is a single file, then convert it to a list
        junc_files = [junc_files]

    #2. run read_junction_files function to read in all junction files
    print(f"Loading files obtained from {sequencing_type} sequencing")

    all_juncs = read_junction_files(junc_files, junc_suffix)

    #3. If gtf_file is not empty, read it in and process it
    if gtf_file is not None:
        gtf_exons_gr = process_gtf(gtf_file)
        print("Done extracting exons from gtf file")
    else:
        pass

    #4. Convert parameters to integers outside the loop
    min_intron = int(min_intron)
    max_intron = int(max_intron)
    min_junc_reads = int(min_junc_reads)
    min_num_cells_wjunc = int(min_num_cells_wjunc)

    #5. Define column names based on sequencing type
    col_names = ["chrom", "chromStart", "chromEnd", "name", "score", "strand", 
             "thickStart", "thickEnd", "itemRgb", "blockCount", "blockSizes", "blockStarts"]
    if sequencing_type == "single_cell":
        col_names += ["num_cells_wjunc", "cell_readcounts"]
    col_names += ["file_name", "cell_type"]
    
    # 6. Clean up junctions and filter for intron length
    all_juncs = clean_up_juncs(all_juncs, col_names, min_intron, max_intron)

    # 7. Make gr object from ALL junctions across all cell types 
    print("Making gr object from all junctions across all cell types")

    juncs_gr = pr.from_dict({"Chromosome": all_juncs["chrom"], "Start": all_juncs["chromStart"], "End": all_juncs["chromEnd"], "Strand": all_juncs["strand"], "Cell": all_juncs["cell_type"], "junction_id": all_juncs["junction_id"], "counts_total": all_juncs["counts_total"]})

    # Unique set of junction coordinates across all samples (or cells) 
    juncs_gr = juncs_gr[["Chromosome", "Start", "End", "Strand", "junction_id", "counts_total"]].drop_duplicate_positions()

    # if min_junc_reads is not none then remove junctions with less than min_junc_reads
    if min_junc_reads is not None:
        juncs_gr = juncs_gr[juncs_gr.counts_total > min_junc_reads]

    #keep only junctions that could be actually related to isoforms that we expect in our cells (via gtf file provided)
    print("The number of junctions prior to assessing distance to exons is " + str(len(juncs_gr.junction_id.unique())))

    # 8. Annotate junctions based on gtf file (if gtf_file is not empty)
    if gtf_file is not None:
        juncs_gr, juncs_coords_unique, clusters = mapping_juncs_exons(juncs_gr, gtf_exons_gr, singleton) 
    else:
        print("Clustering intron splicing events by coordinates")
        juncs_coords_unique = juncs_gr[['Chromosome', 'Start', 'End', 'Strand', 'junction_id', 'counts_total']].drop_duplicate_positions()
        clusters = juncs_coords_unique.cluster(slack=-1, count=True)
        print("The number of clusters after clustering by coordinates is " + str(len(clusters.Cluster.unique())))
        if singleton == False:
            clusters = clusters[clusters.Count > 1]
            # update juncs_gr to include only clusters that are in clusters
            juncs_gr = juncs_gr[juncs_gr.junction_id.isin(clusters.junction_id)]
            juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(clusters.junction_id)]
            print("The number of clusters after removing singletons is " + str(len(clusters.Cluster.unique())))

    # 9. Now for each cluster we want to check that each junction shares a splice site with at least one other junction in the cluster
    if filter_shared_ss == True:
        clusts_keep = filter_junctions_by_shared_splice_sites(clusters.df)
        # update clusters, juncs_gr, and juncs_coords_unique to only include clusters
        clusters = clusters[clusters.Cluster.isin(clusts_keep)]
        juncs_gr = juncs_gr[juncs_gr.junction_id.isin(clusters.junction_id)]

    juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(clusters.junction_id)]
    print("The number of clusters after filtering for shared splice sites is " + str(len(clusters.Cluster.unique())))
    print("The number of junctions after filtering for shared splice sites is " + str(len(juncs_coords_unique.junction_id.unique())))

    # 10. Update our all_juncs file to only include junctions that are part of clusters
    all_juncs = all_juncs[all_juncs.junction_id.isin(juncs_coords_unique.junction_id)]

    # 11. Refine intron clusters based on splice sites found in them -> remove low confidence junctions basically a filter to see which junctions to keep
    print("Refining intron clusters to account for junction usage ratio threshold...")
    juncs_counts = juncs_gr.df[['junction_id', 'Start', 'End', 'counts_total']].drop_duplicates()
    clust_info = clusters.df[['Cluster', 'junction_id']].drop_duplicates()
    clust_info = clust_info.merge(juncs_counts)
    junc_scores_all = refine_clusters(clust_info)
    junc_scores_all = junc_scores_all[junc_scores_all.min_usage >= threshold_inc]
    # add 5ss and 3ss usatio of each junction to all_juncs
    all_juncs = all_juncs.merge(junc_scores_all[['junction_id', 'total_5ss_counts', 'total_3ss_counts', "5SS_usage", "3SS_usage"]], on='junction_id')

    # remove junctions that are in junc_scores_all from juncs_gr, clusters, all_juncs and juncs_coords_unique
    juncs_gr = juncs_gr[juncs_gr.junction_id.isin(junc_scores_all.junction_id)]
    clusters = clusters[clusters.junction_id.isin(junc_scores_all.junction_id)]
    all_juncs = all_juncs[all_juncs.junction_id.isin(junc_scores_all.junction_id)]
    juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(junc_scores_all.junction_id)]
    print("The number of clusters after removing low confidence junctions is " + str(len(clusters.Cluster.unique())))

    # 12. given junctions that remain, see if need to recluster introns (low confidence junctions removed)
    print("Reclustering intron splicing events after low confidence junction removal")
    # check if there are any duplicate entried in pyranges object 
    juncs_gr = juncs_gr.drop_duplicate_positions()
    # drop original cluster column and add new one
    clusters = juncs_gr.cluster(by="gene_id", slack=-1, count=True)
    
    # 13. remove singletons if there are new ones 
    if((singleton) == False):
        clusters = clusters[clusters.Count > 1]
    
    # update juncs_gr to only include junctions that are part of clusters and update juncs_coords_unique to only include junctions that are part of clusters
    juncs_gr = juncs_gr[juncs_gr.junction_id.isin(clusters.junction_id)]
    juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(clusters.junction_id)]
    print("The number of clusters after removing singletons is " + str(len(clusters.Cluster.unique())))

    # 14. After re-clustering above, need to confirm that junctions still share splice sites  
    print("Confirming that junctions in each cluster share splice sites")
    clusts_keep = filter_junctions_by_shared_splice_sites(clusters.df)
    # update clusters, juncs_gr, and juncs_coords_unique to only include clusters
    clusters = clusters[clusters.Cluster.isin(clusts_keep)]
    juncs_gr = juncs_gr[juncs_gr.junction_id.isin(clusters.junction_id)]
    juncs_coords_unique = juncs_coords_unique[juncs_coords_unique.junction_id.isin(clusters.junction_id)]
    all_juncs = all_juncs[all_juncs.junction_id.isin(juncs_coords_unique.junction_id)]
    print("The number of clusters after filtering for shared splice sites is " + str(len(clusters.Cluster.unique())))

    if gtf_file is not None:
        clusts_unique = clusters.df[["Cluster", "junction_id", "gene_id", "Count"]].drop_duplicates()
        juncs_gr = juncs_gr[["Chromosome", "Start", "End", "Strand", "junction_id", "Start_b", "End_b", "gene_id", "gene_name", "transcript_id", "exon_id"]]
        juncs_gr = juncs_gr.drop_duplicate_positions()
        juncs_gr.to_bed(junc_bed_file, chain=True) #add option to add prefix to file name
        print("Saved final list of junction coordinates to " + junc_bed_file)
    else:
        clusts_unique = clusters.df[["Cluster", "junction_id", "Count"]].drop_duplicates()
        juncs_gr = juncs_gr[["Chromosome", "Start", "End", "Strand", "junction_id"]]
        juncs_gr = juncs_gr.drop_duplicate_positions()
        juncs_gr.to_bed(junc_bed_file, chain=True)
        print("Saved final list of junction coordinates to " + junc_bed_file)
    
     # merge juncs_gr with corresponding cluster id
    all_juncs_df = all_juncs.merge(clusts_unique, how="left")

    # get final list of junction coordinates and save to bed file for visualization
    print("The number of clusters to be finally evaluated is " + str(len(all_juncs_df.Cluster.unique()))) 
    print("The number of junctions to be finally evaluated is " + str(len(all_juncs_df.junction_id.unique())))

    # assert unique number of junctions and clusters in all_juncs_df and clusters_df is the same 
    assert len(all_juncs_df.junction_id.unique()) == len(clusters.df.junction_id.unique())
    assert len(all_juncs_df.Cluster.unique()) == len(clusters.df.Cluster.unique()) 
    
    # 12. Save the final list of intron clusters to a file
    # to the output file add the parameters that was used so user can easily tell how they generated this file 
    date = time.strftime("%Y%m%d")
    output = output_file + "_" + str(min_intron) + "_" + str(max_intron) + "_" + str(min_junc_reads) + "_" + date + "_" + str(sequencing_type) 
    with gzip.open(output + '.gz', mode='wt', encoding='utf-8') as f:
        all_juncs_df.to_csv(f, index=False, sep="}")
    print("You can find the output file here: " + output + ".gz")
    print("Finished obtaining intron cluster files!")

    # also return the final list of intron clusters if running in notebook 
    if run_notebook:
        return all_juncs_df

In [3]:
# Define path that contains some junction files (only 2 files are used for this example, corresponding to 2 individual cells)
juncs_path = "/commons/projects/knowles_singlecell_splicing/bulk_whole_mouse/junctions/"
print("The junctions are loaded from the following path: " + juncs_path) 
junc_files = juncs_path

# we provide a gtf file for the human genome as well to make better sense of the junctions that are detected in cells
# please replace with the path to the gtf file on your system
gtf_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf" 

# define additional parameters 
sequencing_type = "bulk"

output_path = "/commons/projects/knowles_singlecell_splicing/bulk_whole_mouse/Leaflet/"
# ensure output files are to be saved in output_path 
output_file =  output_path + "bulk_intron_clusters"
junc_bed_file= output_path + "bulk_juncs.bed" # you can load this file into IGV to visualize the junction coordinates 
min_intron_length = 50
max_intron_length = 500000
threshold_inc = 0.01 
min_junc_reads = 5
keep_singletons = False # ignore junctions that do not share splice sites with any other junction (likely const)
junc_suffix = "*_junctions.bed" # depends on how you ran regtools 

The junctions are loaded from the following path: /commons/projects/knowles_singlecell_splicing/bulk_whole_mouse/junctions/


In [4]:
if type(junc_files) == list:
    # do nothing 
    pass
elif "," in junc_files:
    junc_files = junc_files.split(",")
else:
    # if junc_files is a single file, then convert it to a list
    junc_files = [junc_files]
#2. run read_junction_files function to read in all junction files
print(f"Loading files obtained from {sequencing_type} sequencing")
all_juncs = read_junction_files(junc_files, junc_suffix)

Loading files obtained from bulk sequencing


100%|██████████| 1/1 [00:00<00:00,  2.50it/s]


In [6]:
if gtf_file is not None:
    gtf_exons_gr = process_gtf(gtf_file)
    print("Done extracting exons from gtf file")
else:
    pass
#4. Convert parameters to integers outside the loop
min_intron = int(50)
max_intron = int(50000)
min_junc_reads = int(min_junc_reads)
min_num_cells_wjunc = int(1)

#5. Define column names based on sequencing type
col_names = ["chrom", "chromStart", "chromEnd", "name", "score", "strand", 
         "thickStart", "thickEnd", "itemRgb", "blockCount", "blockSizes", "blockStarts"]
if sequencing_type == "single_cell":
    col_names += ["num_cells_wjunc", "cell_readcounts"]
col_names += ["file_name", "cell_type"]

The gtf file you provided is /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf
Reading the gtf may take a minute...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'havana_gene', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'tag', 'havana_transcript', 'exon_id', 'protein_id', 'ccdsid', 'ont']


Reading gtf file took 85.88 seconds
The number of unique exons is 437943
The number of unique transcript ids is 135995
The number of unique gene ids is 54446
+++++++++++++++++++++++++++++++++++++++++++++++++++++++
Done extracting exons from gtf file


In [7]:
all_juncs = clean_up_juncs(all_juncs, col_names, min_intron, max_intron)
all_juncs.head()

Cleaning up 'chrom' column


,junction_id,counts_total,chrom,chromStart,chromEnd,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts,file_name,cell_type,block_add_start,block_subtract_end,intron_length
0,10_100016035_100051845,31,10,100016035,100051845,JUNC00103083,31,+,100015963,100051918,"255,0,0",2,"72,73","0,35882",/commons/projects/knowles_singlecell_splicing/...,/commons/projects/knowles_singlecell_splicing/...,72,73,35810
1,10_100051959_100064083,56,10,100051959,100064083,JUNC00103084,56,+,100051887,100064146,"255,0,0",2,"72,63","0,12196",/commons/projects/knowles_singlecell_splicing/...,/commons/projects/knowles_singlecell_splicing/...,72,63,12124
2,10_100064146_100076734,44,10,100064146,100076734,JUNC00103085,44,+,100064082,100076806,"255,0,0",2,"64,72","0,12652",/commons/projects/knowles_singlecell_splicing/...,/commons/projects/knowles_singlecell_splicing/...,64,72,12588
3,10_100076905_100079973,61,10,100076905,100079973,JUNC00103086,61,+,100076838,100080045,"255,0,0",2,"67,72","0,3135",/commons/projects/knowles_singlecell_splicing/...,/commons/projects/knowles_singlecell_splicing/...,67,72,3068
4,10_100080130_100080856,38,10,100080130,100080856,JUNC00103087,38,+,100080057,100080927,"255,0,0",2,"73,71","0,799",/commons/projects/knowles_singlecell_splicing/...,/commons/projects/knowles_singlecell_splicing/...,73,71,726


In [8]:
print(f"Number of junctions after first step cleaning up is {len(all_juncs.junction_id.unique())}")

Number of junctions after first step cleaning up is 167764


In [36]:
juncs_gr = pr.from_dict({"Chromosome": all_juncs["chrom"], "Start": all_juncs["chromStart"], "End": all_juncs["chromEnd"], "Strand": all_juncs["strand"], "Cell": all_juncs["cell_type"], "junction_id": all_juncs["junction_id"], "counts_total": all_juncs["counts_total"]})

# Unique set of junction coordinates across all samples (or cells) 
juncs_gr = juncs_gr[["Chromosome", "Start", "End", "Strand", "junction_id", "counts_total"]].drop_duplicate_positions()
juncs_gr

,Chromosome,Start,End,Strand,junction_id,counts_total
0,1,100050789,100072047,+,1_100050789_100072047,6
1,1,100072251,100076029,+,1_100072251_100076029,5
2,1,100141438,100160037,+,1_100141438_100160037,3
3,1,100160187,100164067,+,1_100160187_100164067,6
4,1,100164239,100213666,+,1_100164239_100213666,5
...,...,...,...,...,...,...
167759,Y,90817129,90823873,-,Y_90817129_90823873,1
167760,Y,90822974,90823045,-,Y_90822974_90823045,1
167761,Y,90823066,90823873,-,Y_90823066_90823873,3
167762,Y,90837279,90838776,-,Y_90837279_90838776,9


In [37]:
min_junc_reads=2

In [38]:
# if min_junc_reads is not none then remove junctions with less than min_junc_reads
if min_junc_reads is not None:
    juncs_gr = juncs_gr[juncs_gr.counts_total > min_junc_reads]

In [39]:
print("The number of junctions after filtering by min_junc_reads  " + str(len(juncs_gr.junction_id.unique())))

The number of junctions after filtering by min_junc_reads  128592


In [40]:
gtf_file

'/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf'

In [41]:
juncs_gr, juncs_coords_unique, clusters = mapping_juncs_exons(juncs_gr, gtf_exons_gr, False) 

Annotating junctions with known exons based on input gtf file
The number of junctions after assessing distance to exons is 124846
Clustering intron splicing events by gene_id
The number of clusters after clustering by gene_id is 114895
The number of junctions after removing singletons is 15533
The number of clusters after removing singletons is 5582


In [42]:
clusters

,Chromosome,Start,End,Strand,junction_id,gene_id,Cluster,Count
0,1,36521839,36524033,+,1_36521839_36524033,ENSMUSG00000001138.13,6,3
1,1,36521839,36525165,+,1_36521839_36525165,ENSMUSG00000001138.13,6,3
2,1,36524132,36525165,+,1_36524132_36525165,ENSMUSG00000001138.13,6,3
3,1,150445963,150446084,+,1_150445963_150446084,ENSMUSG00000006005.18,77,2
4,1,150445963,150446123,+,1_150445963_150446123,ENSMUSG00000006005.18,77,2
...,...,...,...,...,...,...,...,...
15528,Y,1013473,1014633,+,Y_1013473_1014633,ENSMUSG00000069049.11,114843,3
15529,Y,90793417,90816348,+,Y_90793417_90816348,ENSMUSG00000096768.7,114856,2
15530,Y,90793680,90816348,+,Y_90793680_90816348,ENSMUSG00000096768.7,114856,2
15531,Y,1168185,1169122,-,Y_1168185_1169122,ENSMUSG00000068457.14,114871,2
